# NB09 — Statistical Robustness and Equivalence Analysis

This notebook harmonizes leakage-safe OOF predictions, ranks the FULL_11 benchmark descriptively, applies Nadeau–Bengio corrected resampled inference, and evaluates practical-equivalence sensitivity. Probability calibration and selective prediction are intentionally handled in NB16 using the final strict outer-training-only protocol.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, platform, warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import t as student_t
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import (
    f1_score, accuracy_score, log_loss
)

warnings.filterwarnings('ignore')

ROOT = Path('/content/drive/MyDrive/DRY_BEAN_HYBRID_Q1')
RESULTS = ROOT/'03_RESULTS'
FIGURES = ROOT/'05_FIGURES'
TABLES = ROOT/'06_TABLES'
LOGS = ROOT/'07_LOGS'

OUT = RESULTS/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
FIG = FIGURES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
TAB = TABLES/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'
LOG = LOGS/'NB09_ROBUSTNESS_CALIBRATION_SELECTIVE'

for p in [OUT, FIG, TAB, LOG]:
    p.mkdir(parents=True, exist_ok=True)

NB06_METRICS = RESULTS/'NB06_STATS'/'all_model_metrics_by_fold.csv'
NB06_OOF = RESULTS/'NB06_STATS'/'all_oof_predictions.csv'
NB08_METRICS = RESULTS/'NB08_EXTENDED_ABLATION'/'nb08_metrics_by_fold.csv'
NB08_OOF = RESULTS/'NB08_EXTENDED_ABLATION'/'nb08_oof_predictions.csv'

for p in [NB06_METRICS, NB06_OOF, NB08_METRICS, NB08_OOF]:
    assert p.exists(), f'Missing required input: {p}'

SEEDS = [2026, 2027, 2028]
OUTER_FOLDS = 5
N_TRAIN = 3200
N_TEST = 800
TEST_TRAIN_RATIO = N_TEST / N_TRAIN

ALPHA = 0.05
PRIMARY_EQUIV_MARGIN = 0.010
EQUIV_MARGINS = [0.005, 0.010, 0.015]  # primary ±0.010; ±0.005 and ±0.015 sensitivity bounds
ROPE_MARGINS = [0.005, 0.010]

print('NB09 output:', OUT)
print('Base-model retraining: NO')


## 1. Load and harmonize OOF predictions

NB06 contains the original FULL_11 models. NB08 contributes the additional FULL_11 comparators and all REDUCED_7 reruns. The combined table is checked so that each model × feature-set × seed has exactly 4000 OOF predictions.


In [ ]:
m6 = pd.read_csv(NB06_METRICS)
o6 = pd.read_csv(NB06_OOF)
m8 = pd.read_csv(NB08_METRICS)
o8 = pd.read_csv(NB08_OOF)

m6['feature_set'] = 'FULL_11'
o6['feature_set'] = 'FULL_11'

metric_cols = [
    'f1_macro','accuracy','balanced_accuracy','precision_macro','recall_macro',
    'mcc','kappa','log_loss','feature_set','model','seed','outer_fold'
]
metrics = pd.concat([
    m6[metric_cols],
    m8[metric_cols]
], ignore_index=True)

oof = pd.concat([o6, o8], ignore_index=True, sort=False)

# Remove accidental duplicate records if a future upstream notebook contains overlap.
key_metric = ['feature_set','model','seed','outer_fold']
metrics = metrics.sort_values(key_metric).drop_duplicates(key_metric, keep='last').reset_index(drop=True)

key_oof = ['feature_set','model','seed','outer_fold','row_id']
oof = oof.sort_values(key_oof).drop_duplicates(key_oof, keep='last').reset_index(drop=True)

prob_cols = [c for c in oof.columns if c.startswith('prob_')]
CLASSES = [c.replace('prob_','',1) for c in prob_cols]

assert len(CLASSES) == 4, CLASSES
assert set(oof['y_true'].unique()) == set(CLASSES)

counts = oof.groupby(['feature_set','model','seed']).size()
assert (counts == 4000).all(), counts[counts != 4000]

fold_counts = oof.groupby(['feature_set','model','seed','outer_fold']).size()
assert (fold_counts == 800).all(), fold_counts[fold_counts != 800]

metrics.to_csv(OUT/'all_metrics_harmonized.csv', index=False)
oof.to_csv(OUT/'all_oof_harmonized.csv', index=False)

print('Classes:', CLASSES)
print('FULL_11 models:', metrics.loc[metrics.feature_set=='FULL_11','model'].nunique())
print('REDUCED_7 models:', metrics.loc[metrics.feature_set=='REDUCED_7','model'].nunique())
print('OOF rows:', len(oof))


## 2. FULL_11 ranking and automatic reference model

The reference model for inferential comparisons is selected **only by mean macro-F1 across the 15 matched outer-test folds**. The ranking is descriptive; statistical comparisons are reported separately below.


In [ ]:
full_metrics = metrics[metrics.feature_set=='FULL_11'].copy()

rank_full = (
    full_metrics.groupby('model')
    .agg(
        mean_macro_f1=('f1_macro','mean'),
        sd_macro_f1=('f1_macro','std'),
        mean_accuracy=('accuracy','mean'),
        mean_balanced_accuracy=('balanced_accuracy','mean'),
        mean_log_loss=('log_loss','mean'),
        n_folds=('f1_macro','size')
    )
    .reset_index()
    .sort_values(['mean_macro_f1','mean_accuracy'], ascending=False)
    .reset_index(drop=True)
)
rank_full['rank_macro_f1'] = np.arange(1, len(rank_full)+1)
rank_full.to_csv(TAB/'full11_model_ranking.csv', index=False)

REFERENCE_MODEL = rank_full.loc[0,'model']
print('Reference model:', REFERENCE_MODEL)
display(rank_full)


## 3. Corrected-resampling inference for FULL_11

For each comparator, fold-level macro-F1 differences are paired to the same seed and outer fold as the reference model. The corrected standard error is

\[
SE_{corr}=\sqrt{\left(\frac{1}{n}+\frac{n_{test}}{n_{train}}\right)s_d^2},
\]

with \(n=15\), \(n_{test}/n_{train}=800/3200=0.25\). Two-sided p-values are Holm-adjusted across reference-vs-comparator tests.


In [ ]:
def corrected_stats(d, test_train_ratio=TEST_TRAIN_RATIO, alpha=ALPHA):
    d = np.asarray(d, dtype=float)
    n = len(d)
    mean_d = float(np.mean(d))
    sd_d = float(np.std(d, ddof=1))
    var_d = sd_d**2
    se_corr = math.sqrt((1/n + test_train_ratio) * var_d)
    df = n - 1
    if se_corr == 0:
        t_stat = np.inf if mean_d > 0 else (-np.inf if mean_d < 0 else 0.0)
        p_two = 0.0 if mean_d != 0 else 1.0
        ci_low = ci_high = mean_d
    else:
        t_stat = mean_d / se_corr
        p_two = 2 * student_t.sf(abs(t_stat), df)
        crit = student_t.ppf(1-alpha/2, df)
        ci_low = mean_d - crit*se_corr
        ci_high = mean_d + crit*se_corr
    return {
        'n_pairs': n,
        'mean_delta_macro_f1': mean_d,
        'sd_delta': sd_d,
        'corrected_se': se_corr,
        'df': df,
        't_corrected': t_stat,
        'p_two_sided': p_two,
        'ci95_low': ci_low,
        'ci95_high': ci_high
    }

ref = (
    full_metrics[full_metrics.model==REFERENCE_MODEL]
    [['seed','outer_fold','f1_macro']]
    .rename(columns={'f1_macro':'f1_ref'})
)

rows = []
for model in rank_full.model:
    if model == REFERENCE_MODEL:
        continue
    cur = (
        full_metrics[full_metrics.model==model]
        [['seed','outer_fold','f1_macro']]
        .rename(columns={'f1_macro':'f1_model'})
    )
    z = ref.merge(cur, on=['seed','outer_fold'], validate='one_to_one')
    assert len(z) == 15
    # Positive delta means comparator > reference.
    d = z.f1_model.values - z.f1_ref.values
    st = corrected_stats(d)
    st['reference_model'] = REFERENCE_MODEL
    st['comparator_model'] = model
    rows.append(st)

corr_tests = pd.DataFrame(rows)
reject, p_holm, _, _ = multipletests(corr_tests.p_two_sided.values, alpha=ALPHA, method='holm')
corr_tests['p_holm'] = p_holm
corr_tests['significant_holm_0.05'] = reject
corr_tests = corr_tests.sort_values('mean_delta_macro_f1', ascending=False).reset_index(drop=True)
corr_tests.to_csv(TAB/'full11_corrected_resampled_tests_vs_reference.csv', index=False)

display(corr_tests)


## 4. Equivalence-margin and correlated-t probability sensitivity

Equivalence is evaluated as a **sensitivity analysis**, not as an arbitrary declaration of practical equivalence. For each pre-specified margin, both one-sided tests must reject at \(lpha=0.05\) using the corrected standard error.

A correlated-t probability summary is also reported for ROPEs of ±0.005 and ±0.010 macro-F1. These probabilities summarize the corrected-t uncertainty model and should not be interpreted as independent replication probabilities.


In [ ]:
def corrected_tost(mean_d, se, df, margin):
    if se == 0:
        equiv = abs(mean_d) < margin
        return 0.0 if equiv else 1.0, 0.0 if equiv else 1.0, 0.0 if equiv else 1.0, equiv
    t_lower = (mean_d + margin) / se   # H0: delta <= -margin
    p_lower = student_t.sf(t_lower, df)
    t_upper = (mean_d - margin) / se   # H0: delta >= +margin
    p_upper = student_t.cdf(t_upper, df)
    p_tost = max(p_lower, p_upper)
    return p_lower, p_upper, p_tost, bool(p_tost < ALPHA)

def correlated_t_probs(mean_d, se, df, rope):
    if se == 0:
        return (
            float(mean_d < -rope),
            float(abs(mean_d) <= rope),
            float(mean_d > rope)
        )
    p_below = student_t.cdf((-rope - mean_d)/se, df)
    p_above = student_t.sf((rope - mean_d)/se, df)
    p_rope = max(0.0, 1.0 - p_below - p_above)
    return p_below, p_rope, p_above

eq_rows = []
prob_rows = []

for _, r in corr_tests.iterrows():
    for margin in EQUIV_MARGINS:
        p1,p2,pt,equiv = corrected_tost(
            r.mean_delta_macro_f1, r.corrected_se, int(r.df), margin
        )
        eq_rows.append({
            'reference_model': REFERENCE_MODEL,
            'comparator_model': r.comparator_model,
            'mean_delta_comparator_minus_reference': r.mean_delta_macro_f1,
            'margin': margin,
            'p_lower': p1,
            'p_upper': p2,
            'p_tost': pt,
            'equivalent_at_alpha_0.05': equiv
        })
    for rope in ROPE_MARGINS:
        pb,pr,pa = correlated_t_probs(
            r.mean_delta_macro_f1, r.corrected_se, int(r.df), rope
        )
        prob_rows.append({
            'reference_model': REFERENCE_MODEL,
            'comparator_model': r.comparator_model,
            'mean_delta_comparator_minus_reference': r.mean_delta_macro_f1,
            'rope': rope,
            'p_comparator_worse_than_rope': pb,
            'p_practically_equivalent_within_rope': pr,
            'p_comparator_better_than_rope': pa
        })

equiv = pd.DataFrame(eq_rows)
corr_prob = pd.DataFrame(prob_rows)

equiv.to_csv(TAB/'full11_equivalence_margin_sensitivity.csv', index=False)
corr_prob.to_csv(TAB/'full11_correlated_t_probability_summary.csv', index=False)

print('Equivalence sensitivity:')
display(equiv)
print('Correlated-t probability summary:')
display(corr_prob)
